# Python RAG 交互式学习笔记

这份 Notebook 是 `python_rag_notes.md` 的配套运行版。它用于复习 RAG 的核心链路：数据加载、文本切片、Embedding 思路、向量检索、Prompt 组装和答案生成。

使用建议：

- 前半部分全部离线可运行，不需要 API key，也不连接外部数据库。
- Milvus、真实 Embedding 和 LLM 调用只保留为扩展方向，具体可参考 `python_milvus_notes.ipynb`。
- 学习时先跑离线示例，理解流程后再替换为真实向量库和模型。

## 1. RAG 的主流程

RAG 是 Retrieval-Augmented Generation：先检索，再生成。

```text
数据准备：文件 -> Loader -> 清洗 -> Splitter -> Chunks -> Embedding -> VectorStore
查询生成：问题 -> Query Embedding -> 检索 -> 相关片段 -> Prompt -> LLM -> Answer
```

RAG 的关键不是“把资料塞给模型”，而是把正确资料以合适粒度取回来，并让模型严格基于资料回答。

In [ ]:
from dataclasses import dataclass


@dataclass
class SimpleDocument:
    page_content: str
    metadata: dict


raw_documents = [
    SimpleDocument(
        "RAG 会先检索知识库，再把相关上下文交给大语言模型回答。",
        {"source": "rag_intro.md", "title": "RAG 原理"},
    ),
    SimpleDocument(
        "Milvus 是向量数据库，常用于语义搜索、相似度检索和企业知识库。",
        {"source": "milvus.md", "title": "Milvus"},
    ),
    SimpleDocument(
        "Text Splitter 会把长文档切成多个文本块，chunk_size 和 overlap 会影响召回质量。",
        {"source": "splitter.md", "title": "文本切片"},
    ),
]

raw_documents

## 2. 文本切片

切片要平衡两个目标：

- 太短：语义不完整，检索回来缺上下文。
- 太长：噪声太多，Prompt 成本高，也可能淹没答案。

常见做法是先按标题、段落等结构切，再按长度兜底。

In [ ]:
def simple_split_text(text: str, chunk_size: int = 32, overlap: int = 8) -> list[str]:
    if chunk_size <= overlap:
        raise ValueError("chunk_size 必须大于 overlap")
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - overlap
    return chunks


long_text = (
    "RAG 项目通常先加载 PDF、Word、Markdown 等文档，然后清洗文本，"
    "再按照标题、段落或固定长度切成文本块。文本块会被 Embedding 模型转成向量，"
    "写入 Milvus、Chroma 或 FAISS 等向量库。用户提问时，系统先检索相关文本块，"
    "再让模型根据上下文生成答案。"
)

chunks = simple_split_text(long_text, chunk_size=45, overlap=10)
for i, chunk in enumerate(chunks, 1):
    print(i, chunk)

## 3. 用关键词重叠模拟检索

真实 RAG 会用 Embedding 向量做相似度检索。为了离线理解流程，这里用字符重叠模拟一个很小的 retriever。

In [ ]:
chunk_docs = [
    SimpleDocument(chunk, {"source": "rag_pipeline.md", "chunk": i})
    for i, chunk in enumerate(chunks, 1)
]
chunk_docs.extend(raw_documents)


def keyword_retrieve(question: str, documents: list[SimpleDocument], k: int = 3) -> list[tuple[int, SimpleDocument]]:
    query_terms = set(question.lower())
    scored = []
    for doc in documents:
        score = len(query_terms & set(doc.page_content.lower()))
        scored.append((score, doc))
    scored.sort(key=lambda item: item[0], reverse=True)
    return [(score, doc) for score, doc in scored[:k] if score > 0]


question = "RAG 为什么需要切片和向量库？"
for score, doc in keyword_retrieve(question, chunk_docs):
    print(score, doc.metadata, doc.page_content)

## 4. 组装 RAG Prompt

Prompt 中应明确三件事：

1. 只能根据上下文回答。
2. 上下文没有答案时说不知道。
3. 尽量返回来源，便于追溯。

In [ ]:
def build_rag_prompt(question: str, retrieved: list[tuple[int, SimpleDocument]]) -> str:
    context = []
    for score, doc in retrieved:
        source = doc.metadata.get("source", "unknown")
        chunk = doc.metadata.get("chunk", "-")
        context.append(f"[source={source}, chunk={chunk}, score={score}] {doc.page_content}")

    return "\n".join(
        [
            "你是一个严谨的 RAG 助手。",
            "请只根据给定上下文回答；如果上下文没有答案，就说不知道。",
            "",
            "上下文：",
            *context,
            "",
            f"问题：{question}",
        ]
    )


retrieved = keyword_retrieve(question, chunk_docs)
print(build_rag_prompt(question, retrieved))

## 5. 模拟生成答案

真实项目中，最后一步会调用 LLM。这里为了离线运行，用一个简单函数模拟“基于上下文回答”。

In [ ]:
def fake_llm_answer(question: str, retrieved: list[tuple[int, SimpleDocument]]) -> str:
    if not retrieved:
        return "不知道。当前知识库没有检索到足够相关的上下文。"
    sources = ", ".join(str(doc.metadata.get("source")) for _, doc in retrieved)
    return (
        "RAG 需要切片，是为了把长文档拆成适合检索和放入 Prompt 的文本块；"
        "需要向量库，是为了保存文本块向量并根据用户问题做相似度检索。"
        f"\n来源：{sources}"
    )


print(fake_llm_answer(question, retrieved))

## 6. 可选：LangChain Text Splitter

如果已安装 `langchain-text-splitters`，可以使用 `RecursiveCharacterTextSplitter`。它会按分隔符优先级递归切割，比手写固定长度更适合学习项目。

In [ ]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=60,
        chunk_overlap=12,
        separators=["\n\n", "\n", "。", "，", ""],
    )
    lc_chunks = splitter.split_text(long_text)
    for i, chunk in enumerate(lc_chunks, 1):
        print(i, chunk)
except ImportError:
    print("未安装 langchain-text-splitters，跳过 LangChain splitter 示例。")

## 7. Milvus RAG 实战位置

Milvus 在 RAG 中通常作为 VectorStore：

```text
Document chunks -> Embedding -> Milvus collection
User query -> Query Embedding -> Milvus search -> Prompt context
```

完整的 Milvus 运行、collection、schema、索引、过滤搜索、LangChain Milvus Retriever 示例，放在 `python_milvus_notes.ipynb`。

## 8. Milvus dense/sparse 字段设计

当前本机 Docker 里启动 Milvus，只代表数据库服务可连接，不代表里面已经有 RAG collection、dense 字段、sparse 字段或文档数据。RAG 真正要用 Milvus，需要执行：

```text
启动 Milvus 服务
-> 创建 collection schema
-> 定义 text / dense / sparse / metadata 字段
-> 为 dense 和 sparse 建索引
-> 解析文档并切片
-> 生成 dense embedding
-> 写入 text、dense、metadata
-> Milvus 根据 BM25 Function 生成 sparse
-> 查询时做 dense + sparse 混合检索
```

常见字段设计：

| 字段 | Milvus 类型 | 用途 |
| --- | --- | --- |
| `id` | `INT64` 或 `VARCHAR` | 主键，定位 chunk |
| `text` | `VARCHAR` | chunk 原文，也可用于 BM25 分词 |
| `dense` | `FLOAT_VECTOR` | 稠密语义向量，例如 BGE、Qwen、OpenAI Embedding |
| `sparse` | `SPARSE_FLOAT_VECTOR` | 稀疏关键词向量，例如 BM25 Function 输出 |
| `source` | `VARCHAR` | 文件名、URL、知识来源 |
| `page` | `INT64` | PDF 页码或文档页码 |
| `title` | `VARCHAR` | 标题路径或章节名 |
| `tenant_id` | `VARCHAR` | 多租户或权限过滤 |

简单记忆：

```text
text   = 原文
dense  = 语义相似
sparse = 关键词匹配
metadata = 来源、页码、权限、分类
```

PyMilvus schema 示例：

```python
from pymilvus import DataType, Function, FunctionType, MilvusClient

client = MilvusClient(uri="http://127.0.0.1:19530")

schema = client.create_schema(auto_id=True)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(
    field_name="text",
    datatype=DataType.VARCHAR,
    max_length=4000,
    enable_analyzer=True,
)
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="source", datatype=DataType.VARCHAR, max_length=500)
schema.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=500)
schema.add_field(field_name="page", datatype=DataType.INT64)

bm25_function = Function(
    name="text_bm25_emb",
    input_field_names=["text"],
    output_field_names=["sparse"],
    function_type=FunctionType.BM25,
)
schema.add_function(bm25_function)
```

索引示例：

```python
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="dense",
    index_name="dense_vector_index",
    index_type="HNSW",
    metric_type="IP",
    params={"M": 16, "efConstruction": 64},
)

index_params.add_index(
    field_name="sparse",
    index_name="sparse_inverted_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={
        "inverted_index_algo": "DAAT_MAXSCORE",
        "bm25_k1": 1.6,
        "bm25_b": 0.75,
    },
)
```

写入时通常由应用侧生成 `dense`，Milvus 根据 BM25 Function 自动生成 `sparse`：

```python
row = {
    "text": "Milvus 支持 dense+sparse 混合检索，适合 RAG 知识库。",
    "dense": embedding_model.embed_query("Milvus 支持 dense+sparse 混合检索，适合 RAG 知识库。"),
    "source": "python_rag_notes.ipynb",
    "title": "Milvus dense/sparse 字段设计",
    "page": 1,
}

client.insert(collection_name="rag_docs", data=[row])
```



## 8.1 DeepSeek 和 Embedding 的分工

`OpenAIEmbeddings` 需要配置 `OPENAI_API_KEY`，因为它调用的是 OpenAI 的 embedding 接口。DeepSeek 的 OpenAI 兼容接口主要用于 Chat / Completion 生成回答；截至本文整理时，DeepSeek 官方 API 文档没有提供独立 embedding 模型。因此 RAG 里不要把 `OpenAIEmbeddings` 直接改成 DeepSeek。

推荐分工：

| 环节 | 推荐选择 | 是否需要 DeepSeek Key |
| --- | --- | --- |
| 文档向量化 | BGE、Qwen Embedding、Jina、公司统一 embedding 服务 | 不需要 |
| 查询向量化 | 与文档相同的 embedding 模型 | 不需要 |
| 关键词检索 | Milvus BM25 Function / sparse 字段 | 不需要 |
| 答案生成 | DeepSeek Chat，例如 `ChatDeepSeek` | 需要 `DEEPSEEK_API_KEY` |

典型本地学习链路：

```text
本地 embedding 模型 -> Milvus 检索 -> DeepSeek 根据检索上下文生成答案
```

如果公司内部网关额外提供 OpenAI-compatible embedding 接口，可以继续用 `OpenAIEmbeddings(base_url=..., api_key=..., model=...)`，但要确认该网关真的支持 `/embeddings`，不能只看它支持 Chat。

## 9. 最新 RAG 检索方式

现在的 RAG 检索不建议只做“单路向量 top_k”。更稳的做法是把语义检索、关键词检索、过滤、融合、重排和上下文增强组合起来。

### 1. Dense Vector Search

```text
query -> embedding -> dense vector search -> top_k chunks
```

适合自然语言语义问答；不擅长型号、编号、人名、错误码等精确词。

### 2. Full-Text Search / BM25

Milvus 可以通过 analyzer 和 BM25 Function 从 `text` 字段生成 `sparse` 向量，用于全文关键词检索。

```text
text -> analyzer -> BM25 Function -> sparse vector
query text -> BM25 scoring -> keyword-ranked results
```

适合产品型号、订单号、法规条款、API 名、错误码等精确匹配。

### 3. Dense + Sparse Hybrid Search

混合检索是 RAG 项目的常用主线：dense 负责语义，sparse/BM25 负责关键词，再用 ranker 融合。

```text
query
-> dense embedding -> dense search
-> BM25 / sparse embedding -> sparse search
-> WeightedRanker / RRFRanker
-> top candidates
-> reranker
```

常见策略：

| 策略 | 说明 | 适合场景 |
| --- | --- | --- |
| Dense + BM25 | dense 走语义，BM25 走全文关键词 | 企业知识库、课程笔记、产品文档 |
| Dense + Sparse Embedding | sparse 由 BGE-M3/SPLADE 等模型生成 | 对关键词和语义都要求高 |
| WeightedRanker | 给各路召回结果配置权重 | 已知某一路更可靠 |
| RRFRanker | 按排名融合，不强依赖分数尺度 | 多路检索分数不可直接比较 |

推荐默认流程：

```text
dense top_k=20
sparse top_k=20
fusion top_k=20
rerank top_k=5
LLM answer with sources
```

### 4. 多向量 / 多模态混合检索

如果资料里同时有文本、图片、表格、页面截图，可以在一个 collection 中放多个向量字段：

| 字段 | 用途 |
| --- | --- |
| `text_dense` | 文本语义向量 |
| `text_sparse` | BM25 或 sparse embedding |
| `image_dense` | 图片或页面截图向量 |
| `table_dense` | 表格摘要向量 |

适合 PDF 页面含图表、产品图片、PPT、扫描件、合同版面问答。

### 5. Metadata Filter

过滤检索通常和 dense/sparse 一起使用：

```text
query -> dense/sparse search + tenant_id/source/date/category filter
```

典型字段：`tenant_id`、`department`、`source`、`doc_id`、`created_at`、`version`、`category`。权限过滤必须在检索层强制加入，不能只靠 prompt。

### 6. Range Search / Score Threshold

Range Search 用相似度或距离范围过滤结果，相当于“只返回达到可信水位线的结果”。它可以避免无关内容进入 Prompt。

### 7. 分组检索与去重

RAG 很容易出现同一个文档的多个相邻 chunk 占满 top_k。常见做法：

- 按 `doc_id` 分组，每个文档最多返回 N 条。
- 对相邻 chunk 做合并或去重。
- 小 chunk 负责检索，parent chunk 负责生成。

### 8. Reranker 精排

两阶段检索：

```text
dense/sparse/hybrid recall top 20-100
-> reranker(query, candidate)
-> top 3-8
-> LLM
```

适合 top 20 有正确答案但 top 3 不稳定、文档相似度高、hybrid search 后需要统一排序的场景。

### 9. Contextual Retrieval 上下文增强检索

Contextual Retrieval 的核心是在入库前给每个 chunk 增加“它在原文里的上下文说明”，再做 embedding 和 BM25。

```text
原始 chunk
-> LLM 生成上下文摘要：这段在文档中讲什么、属于哪个章节、关键实体是什么
-> context + chunk 一起索引
-> dense + BM25 + rerank
```

### 10. 当前推荐升级路线

```text
阶段 1：dense vector search
阶段 2：dense search + metadata filter
阶段 3：dense + BM25 hybrid search
阶段 4：hybrid search + reranker
阶段 5：contextual retrieval + hybrid search + reranker
阶段 6：多向量 / 多模态 hybrid search
```

企业 RAG 默认链路：

```text
query rewrite
-> metadata permission filter
-> dense vector search
-> BM25 full-text search
-> RRFRanker / WeightedRanker fusion
-> reranker
-> score threshold
-> parent chunk / neighboring chunk expansion
-> LLM answer with citations
```



## 10. 高级 RAG 复习清单

| 方向 | 核心用途 |
| --- | --- |
| Query Rewrite | 改写用户问题，提高召回 |
| Multi Query | 多角度生成查询，合并结果 |
| Hybrid Search | 稠密向量 + BM25 / 稀疏向量 |
| Rerank | 对候选片段重新排序 |
| Corrective RAG | 判断检索质量，不足时修正 |
| Adaptive RAG | 按问题复杂度选择策略 |
| Agentic RAG | 由 Agent 决定是否检索、调用工具或追问 |
| Multimodal RAG | 文本、图片、表格等多模态资料联合检索 |


## 11. 什么样的 RAG 项目更吸引面试官

面试官通常不只看“能不能问答”，而是看你有没有解决真实 RAG 工程问题。

| 层级 | 可讲亮点 |
| --- | --- |
| 数据解析 | PDF、图片、PPT、表格、公式、OCR、版面分析 |
| 切片策略 | 标题递归切片、语义切片、Parent-Child、保留标题路径 |
| 检索策略 | Dense + BM25、Hybrid Search、metadata filter、score threshold |
| 重排策略 | BGE/Qwen reranker、cross-encoder 精排 |
| 生成策略 | 引用来源、拒答、结构化输出、上下文压缩 |
| 评估体系 | Recall@K、MRR、NDCG、Faithfulness、端到端准确率 |
| 工程化 | 多租户权限、缓存、异步入库、日志追踪、成本控制 |

一句话模板：

```text
+我做的不是简单 PDF 问答，而是一个带解析、切片、混合检索、重排、引用和评估闭环的 RAG 系统。
+```

## 12. 上下文增强 Contextual Retrieval

传统 RAG 的常见问题：

1. 固定切片导致上下文丢失，chunk 离开原文后语义漂移。
2. 文档名、章节、年份、产品型号等关键元信息没有进入 chunk，导致检索失败。
3. 单一向量检索不擅长精确关键词，单一 BM25 又不懂语义。

上下文增强的做法：

```text
原始文档 + 当前 chunk
-> 为 chunk 生成上下文描述
-> 上下文描述 + chunk 正文
-> embedding / BM25 / metadata index
```

实际项目里可以给每个 chunk 增加：文档标题、章节路径、页码、主题摘要、关键实体、适用时间范围。这样检索时更容易命中，生成时也更少误解。

## 13. Agentic RAG

Agentic RAG 可以理解为：

```text
RAG + Agent = Agentic RAG
```

传统 RAG 是线性流程：检索一次，生成一次。Agentic RAG 让 Agent 参与检索决策：

- 是否需要检索。
- 是否需要改写 query。
- 是否拆成多个子问题。
- 选择向量库、BM25、图谱、Web、SQL、API 中的哪个工具。
- 检索结果是否足够。
- 不足时继续检索、换策略或追问。

适合多跳问题、多知识源问题、实时工具问题和需要答案自检的问题。不适合简单 FAQ 或极低延迟场景。

## 14. 多模态 RAG

多模态 RAG 处理的是 PDF、扫描件、图片、PPT、表格、图表、音频等资料。

```text
PDF / 图片 / PPT
-> OCR / Layout Parser / 多模态模型
-> 文本、图片、表格、公式结构化
-> 文本 embedding + 图像 embedding / 多模态 embedding
-> Milvus 多向量字段或多 collection
-> hybrid search + rerank
-> 多模态 LLM 生成答案
```

常见方向：

| 类型 | 例子 |
| --- | --- |
| OCR / 文档解析 | PaddleOCR、Dolphin、Dots.OCR、DeepSeek-OCR |
| 多模态 embedding | GME-Qwen2-VL、CLIP、ColPali、ColQwen |
| Reranker | BGE Reranker、Qwen Reranker、Cohere Rerank |
| 多模态推理 | Qwen-VL、GLM-4V、GPT-4o/4.1 视觉模型 |

## 15. Reranker 与两阶段检索

RAG 常见两阶段检索：

```text
第一阶段：向量/BM25 快速召回 top 20-100
第二阶段：reranker 对 query-document pair 精排，选 top 3-8 给 LLM
```

Embedding 检索速度快，适合大规模召回；Reranker 更慢，但能更细地判断“这个片段是否真的回答了这个问题”。

需要 reranker 的典型情况：

- top 20 里有答案，但 top 3 不稳定。
- 文档相似段落很多。
- 问题包含多个条件。
- hybrid search 之后需要统一排序。

## 16. RAG 评估指标

| 指标 | 衡量什么 |
| --- | --- |
| Recall@K | 正确片段是否进入前 K 个结果 |
| Precision@K | 前 K 个结果里相关片段比例 |
| MRR | 第一个正确结果排得多靠前 |
| NDCG | 多个相关结果的排序质量 |
| Faithfulness | 回答是否被上下文支持 |
| Answer Correctness | 最终答案是否正确 |
| Citation Accuracy | 引用是否真的支持答案 |
| Latency / Cost | 延迟和成本 |

面试里要强调：检索评估和生成评估要分开做。先证明召回命中，再证明模型基于命中的上下文答对。

## 17. 实战：PDF / Markdown -> 多策略切分 -> Milvus Hybrid Search

这一节是一个真正可运行的 RAG 入库和检索案例。它演示：

1. 准备 Markdown 和 PDF 文档。
2. 加载 `.md` / `.pdf` 文件并保留来源、页码等 metadata。
3. 先按标题切分，再按长度切分。
4. 使用阿里云百炼 `text-embedding-v4` 生成 dense 向量。
5. 在 Milvus 中创建 `dense` + `sparse` 字段，`sparse` 由 BM25 Function 自动生成。
6. 分别执行 dense search、sparse/BM25 search、hybrid search。

注意：本节会在本地 Milvus 中创建一个新的演示 collection，并插入少量测试文档；不会删除、清空或重建已有 collection。

In [ ]:
from pathlib import Path
from textwrap import dedent

RAG_DATA_DIR = Path("docs/rag_demo_sources")
RAG_DATA_DIR.mkdir(parents=True, exist_ok=True)

md_path = RAG_DATA_DIR / "approval_rag_guide.md"
md_path.write_text(
    dedent(
        """
        # 企业审批 RAG 知识库指南

        ## 1. 场景目标

        企业审批助手需要根据制度文档回答员工问题。系统必须先检索知识库，再基于检索到的制度条款回答，不能凭空编造。

        ## 2. 文档处理流程

        文档进入知识库前，需要解析 PDF、Word、Markdown 和网页。解析后的文本要保留标题、页码、来源文件和更新时间。

        ## 3. 切分策略

        长文档不能只按固定长度切分。推荐先按标题切分，保留章节语义；如果章节仍然过长，再按 chunk_size 和 overlap 做二次切分。

        ## 4. 检索策略

        审批制度里有报销编号、项目编码、员工姓名和制度条款号。仅用 dense embedding 容易漏掉精确词，所以推荐 dense + BM25 混合检索。

        ## 5. 答案生成

        LLM 只能使用检索上下文回答，并且要返回来源。上下文不足时应该回答不知道，而不是编造制度。
        """
    ).strip(),
    encoding="utf-8",
)

print("Markdown sample:", md_path)


In [ ]:
# 可选：生成一个 PDF 示例文件。
# 如果 reportlab 没安装，会跳过 PDF 生成；后面的加载器仍然可以处理你自己放进目录的 PDF。
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.pdfgen import canvas

    pdf_path = RAG_DATA_DIR / "approval_policy_sample.pdf"
    font_name = "Helvetica"
    # Windows 常见中文字体。找不到时退回 Helvetica，中文可能显示为方框，但不影响演示代码结构。
    for font_file, candidate in [
        (r"C:\Windows\Fonts\msyh.ttc", "MicrosoftYaHei"),
        (r"C:\Windows\Fonts\simsun.ttc", "SimSun"),
    ]:
        if Path(font_file).exists():
            pdfmetrics.registerFont(TTFont(candidate, font_file))
            font_name = candidate
            break

    c = canvas.Canvas(str(pdf_path), pagesize=A4)
    c.setFont(font_name, 12)
    lines = [
        "# 差旅报销制度",
        "## 适用范围",
        "本制度适用于正式员工的国内差旅报销。报销单必须关联项目编码。",
        "## 票据要求",
        "交通票、酒店发票和审批单必须一致。缺少发票时，需要上传情况说明。",
        "## 审批规则",
        "单次报销超过 5000 元需要部门负责人和财务负责人双重审批。",
        "## 检索建议",
        "制度编号、金额阈值、项目编码适合 BM25；问题语义适合 dense embedding。",
    ]
    y = 800
    for line in lines:
        c.drawString(60, y, line)
        y -= 28
    c.save()
    print("PDF sample:", pdf_path)
except Exception as exc:
    print("跳过 PDF 示例生成：", type(exc).__name__, exc)


### 17.1 加载 Markdown / PDF

这里不用复杂框架，直接写两个小加载器：

- Markdown：按全文读取，保留文件名。
- PDF：用 `pypdf.PdfReader` 按页提取，保留页码。

真实项目里还可以替换为 Unstructured、MinerU、PaddleOCR、Dots.OCR 等解析器。

In [ ]:
from dataclasses import dataclass, field
from pypdf import PdfReader


@dataclass
class LoadedDocument:
    text: str
    metadata: dict = field(default_factory=dict)


def load_markdown(path: Path) -> LoadedDocument:
    return LoadedDocument(
        text=path.read_text(encoding="utf-8"),
        metadata={"source": path.name, "path": str(path), "file_type": "markdown"},
    )


def load_pdf(path: Path) -> list[LoadedDocument]:
    reader = PdfReader(str(path))
    docs = []
    for page_index, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            docs.append(
                LoadedDocument(
                    text=text,
                    metadata={
                        "source": path.name,
                        "path": str(path),
                        "file_type": "pdf",
                        "page": page_index,
                    },
                )
            )
    return docs


def load_documents_from_dir(directory: Path) -> list[LoadedDocument]:
    loaded = []
    for path in sorted(directory.glob("*")):
        if path.suffix.lower() in {".md", ".markdown"}:
            loaded.append(load_markdown(path))
        elif path.suffix.lower() == ".pdf":
            loaded.extend(load_pdf(path))
    return loaded


raw_loaded_docs = load_documents_from_dir(RAG_DATA_DIR)
print("loaded docs:", len(raw_loaded_docs))
for doc in raw_loaded_docs:
    print(doc.metadata, doc.text[:80].replace("\n", " "))


### 17.2 先按标题切分

标题切分的目标不是控制长度，而是先保住章节语义。这里同时支持 Markdown 标题和从 PDF 中抽取出来的 `#` / `##` 标题行。

In [ ]:
import re

HEADING_PATTERN = re.compile(r"^(#{1,6})\s+(.+?)\s*$")


def split_by_headings(doc: LoadedDocument) -> list[LoadedDocument]:
    sections = []
    current_title = doc.metadata.get("title", "")
    current_lines = []

    def flush():
        if current_lines:
            text = "\n".join(current_lines).strip()
            if text:
                meta = dict(doc.metadata)
                meta["title"] = current_title or meta.get("source", "untitled")
                meta["split_stage"] = "heading"
                sections.append(LoadedDocument(text=text, metadata=meta))

    for raw_line in doc.text.splitlines():
        line = raw_line.strip()
        match = HEADING_PATTERN.match(line)
        if match:
            flush()
            current_title = match.group(2).strip()
            current_lines = [line]
        else:
            current_lines.append(raw_line)
    flush()
    return sections


heading_sections = []
for doc in raw_loaded_docs:
    heading_sections.extend(split_by_headings(doc))

print("heading sections:", len(heading_sections))
for section in heading_sections[:5]:
    print(section.metadata.get("source"), "|", section.metadata.get("title"), "|", len(section.text))


### 17.3 再按长度切分

标题段落可能仍然很长，所以再做长度切分。真实项目可以按 token 切；学习示例里按字符切，核心参数是：

- `chunk_size`：每块最大长度。
- `chunk_overlap`：相邻块重叠长度，避免答案跨块丢失。
- `chunk_index`：同一章节内的块编号。

In [ ]:
def split_by_length(section: LoadedDocument, chunk_size: int = 220, chunk_overlap: int = 60) -> list[LoadedDocument]:
    text = re.sub(r"\n{3,}", "\n\n", section.text.strip())
    if len(text) <= chunk_size:
        meta = dict(section.metadata)
        meta.update({"split_stage": "heading+length", "chunk_index": 0})
        return [LoadedDocument(text=text, metadata=meta)]

    chunks = []
    start = 0
    chunk_index = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk_text = text[start:end].strip()
        if chunk_text:
            meta = dict(section.metadata)
            meta.update({
                "split_stage": "heading+length",
                "chunk_index": chunk_index,
                "chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap,
            })
            chunks.append(LoadedDocument(text=chunk_text, metadata=meta))
            chunk_index += 1
        if end >= len(text):
            break
        start = max(0, end - chunk_overlap)
    return chunks


rag_chunks = []
for section in heading_sections:
    rag_chunks.extend(split_by_length(section, chunk_size=220, chunk_overlap=60))

print("final chunks:", len(rag_chunks))
for chunk in rag_chunks[:6]:
    print("---")
    print(chunk.metadata)
    print(chunk.text[:160].replace("\n", " "))


### 17.4 创建 Milvus dense + sparse collection

这一节使用：

- `dense`：阿里云百炼 `text-embedding-v4` 生成的 1024 维向量。
- `sparse`：Milvus BM25 Function 从 `text` 字段自动生成。
- `source/title/page/chunk_index`：用于来源追踪和过滤。

为了保护已有数据，collection 名带时间戳，每次运行都会创建一个新的演示 collection。

In [ ]:
from datetime import datetime
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from pymilvus import DataType, Function, FunctionType, MilvusClient

load_dotenv(Path.cwd() / ".env")

RAG_MILVUS_URI = "http://localhost:19530"
RAG_COLLECTION = f"learnone_pdf_md_hybrid_{datetime.now():%Y%m%d_%H%M%S}"
DENSE_DIM = int(os.getenv("DASHSCOPE_EMBEDDING_DIMENSIONS", "1024"))

client = MilvusClient(uri=RAG_MILVUS_URI)

schema = client.create_schema(auto_id=True, enable_dynamic_field=False)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=4000, enable_analyzer=True)
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=DENSE_DIM)
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="source", datatype=DataType.VARCHAR, max_length=500)
schema.add_field(field_name="path", datatype=DataType.VARCHAR, max_length=1000)
schema.add_field(field_name="file_type", datatype=DataType.VARCHAR, max_length=50)
schema.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=500)
schema.add_field(field_name="page", datatype=DataType.INT64)
schema.add_field(field_name="chunk_index", datatype=DataType.INT64)

bm25_function = Function(
    name="text_bm25_emb",
    input_field_names=["text"],
    output_field_names=["sparse"],
    function_type=FunctionType.BM25,
)
schema.add_function(bm25_function)

index_params = client.prepare_index_params()
index_params.add_index(
    field_name="dense",
    index_name="dense_vector_index",
    index_type="FLAT",
    metric_type="COSINE",
)
index_params.add_index(
    field_name="sparse",
    index_name="sparse_inverted_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={"inverted_index_algo": "DAAT_MAXSCORE"},
)

client.create_collection(
    collection_name=RAG_COLLECTION,
    schema=schema,
    index_params=index_params,
)

print("created collection:", RAG_COLLECTION)


### 17.5 生成 dense 向量并写入 Milvus

阿里云百炼 embedding 通过 OpenAI 兼容接口调用。关键参数：

- `base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"`
- `model="text-embedding-v4"`
- `dimensions=1024`
- `check_embedding_ctx_length=False`，避免 LangChain 把输入转成 token id 列表。

In [ ]:
dashscope_api_key = os.getenv("DASHSCOPE_API_KEY")
if not dashscope_api_key:
    raise RuntimeError("请先在项目根目录 .env 中配置 DASHSCOPE_API_KEY")

embedding_model = OpenAIEmbeddings(
    model=os.getenv("DASHSCOPE_EMBEDDING_MODEL", "text-embedding-v4"),
    api_key=dashscope_api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    dimensions=DENSE_DIM,
    check_embedding_ctx_length=False,
    chunk_size=16,
)

texts = [chunk.text for chunk in rag_chunks]
dense_vectors = embedding_model.embed_documents(texts)

rows = []
for chunk, vector in zip(rag_chunks, dense_vectors):
    meta = chunk.metadata
    rows.append(
        {
            "text": chunk.text,
            "dense": vector,
            "source": str(meta.get("source", "")),
            "path": str(meta.get("path", "")),
            "file_type": str(meta.get("file_type", "")),
            "title": str(meta.get("title", "")),
            "page": int(meta.get("page", 0) or 0),
            "chunk_index": int(meta.get("chunk_index", 0) or 0),
        }
    )

insert_result = client.insert(collection_name=RAG_COLLECTION, data=rows)
client.flush(collection_name=RAG_COLLECTION)
client.load_collection(collection_name=RAG_COLLECTION)

print("inserted:", insert_result["insert_count"])
print("dense dim:", len(dense_vectors[0]))


### 17.6 Dense Search：语义检索

Dense search 适合“意思相近但措辞不同”的问题。

In [ ]:
query = "超过金额阈值的报销需要谁审批？"
query_dense = embedding_model.embed_query(query)

dense_results = client.search(
    collection_name=RAG_COLLECTION,
    data=[query_dense],
    anns_field="dense",
    search_params={"metric_type": "COSINE", "params": {}},
    limit=3,
    output_fields=["text", "source", "title", "page", "chunk_index"],
)

for hit in dense_results[0]:
    print("score:", round(hit["distance"], 4), "|", hit["entity"]["source"], "|", hit["entity"]["title"])
    print(hit["entity"]["text"][:180].replace("\n", " "))
    print("---")


### 17.7 Sparse / BM25 Search：关键词检索

BM25 适合制度编号、金额、专有名词、项目编码、标题关键词等精确匹配。Milvus 会根据 `text` 字段和 BM25 Function 自动生成 sparse 查询表示。

In [ ]:
sparse_query = "5000 元 双重审批"

sparse_results = client.search(
    collection_name=RAG_COLLECTION,
    data=[sparse_query],
    anns_field="sparse",
    search_params={"metric_type": "BM25", "params": {}},
    limit=3,
    output_fields=["text", "source", "title", "page", "chunk_index"],
)

for hit in sparse_results[0]:
    print("score:", round(hit["distance"], 4), "|", hit["entity"]["source"], "|", hit["entity"]["title"])
    print(hit["entity"]["text"][:180].replace("\n", " "))
    print("---")


### 17.8 Hybrid Search：dense + sparse 融合

Hybrid search 同时发起 dense 和 sparse 两路召回，然后用 ranker 融合。这里优先使用 `RRFRanker`，因为它不要求 dense 分数和 BM25 分数在同一尺度。

In [ ]:
from pymilvus import AnnSearchRequest, RRFRanker

hybrid_query = "报销超过 5000 元时审批规则是什么？"
hybrid_dense = embedding_model.embed_query(hybrid_query)

req_dense = AnnSearchRequest(
    data=[hybrid_dense],
    anns_field="dense",
    param={"metric_type": "COSINE", "params": {}},
    limit=5,
)
req_sparse = AnnSearchRequest(
    data=[hybrid_query],
    anns_field="sparse",
    param={"metric_type": "BM25", "params": {}},
    limit=5,
)

hybrid_results = client.hybrid_search(
    collection_name=RAG_COLLECTION,
    reqs=[req_dense, req_sparse],
    ranker=RRFRanker(),
    limit=3,
    output_fields=["text", "source", "title", "page", "chunk_index"],
)

for hit in hybrid_results[0]:
    print("score:", round(hit["distance"], 4), "|", hit["entity"]["source"], "|", hit["entity"]["title"])
    print(hit["entity"]["text"][:220].replace("\n", " "))
    print("---")


### 17.9 带来源的 Prompt 组装

检索结果不要直接丢给模型。通常要格式化成带来源、标题、页码的上下文，方便模型引用，也方便用户追溯。

In [ ]:
def format_hits_for_prompt(hits) -> str:
    blocks = []
    for i, hit in enumerate(hits, start=1):
        entity = hit["entity"]
        source = entity.get("source", "")
        title = entity.get("title", "")
        page = entity.get("page", 0)
        text = entity.get("text", "")
        blocks.append(f"[来源 {i}] source={source}, title={title}, page={page}\n{text}")
    return "\n\n".join(blocks)

context = format_hits_for_prompt(hybrid_results[0])
question = "报销超过 5000 元时，需要哪些审批？"

prompt = f"""
你是企业制度问答助手。只能根据给定上下文回答；如果上下文没有答案，就说不知道。
回答必须包含来源文件和标题。

上下文：
{context}

问题：{question}
""".strip()

print(prompt)


### 17.10 这个案例在真实项目中的升级点

- PDF 解析：扫描件需要 OCR，复杂 PDF 需要版面解析。
- 切分：标题切分后可以增加语义切分、表格单独切分、parent-child chunk。
- 检索：生产常用 dense + BM25 + metadata filter + reranker。
- 权限：检索时必须加 `tenant_id`、部门、文档权限过滤。
- 评估：用 Recall@K、MRR、Faithfulness、引用准确率持续评估。
- 数据更新：文档变更后要按 `doc_id` 做增量更新，而不是混入旧版本。